# Exceedance probabilities

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/L16B06")

## Load training and test datasets

In [ ]:
import xarray as xr

## Open a netCDF file in a xarray dataset
fname = 'data/garachico256.ens.nc'
ds    = xr.open_dataset(fname)
train = ds['tephra_col_mass']

fname = 'data/garachico2048.ens.nc'
ds    = xr.open_dataset(fname)
test  = ds['tephra_col_mass']

## Load a pre-trained VAE

In [ ]:
import torch
from modules.model import VariationalAutoencoder
from modules.dataset import MinMaxScale

## Load weight parameters and some metadata
fname = output_dir / 'model.pt'
checkpoint = torch.load(fname)

## Recreate the model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'])
model.load_state_dict(checkpoint['model_state_dict'])

## Generate a VAE ensemble

In [ ]:
## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
transform = MinMaxScale(min_value, max_value)

## Generate nens new samples
nens = 5000
z = torch.randn(nens, checkpoint['LATENT_DIM'])
with torch.no_grad():
    new_sample = model.decode(z)
    x = transform.invert(new_sample).squeeze()

## Plot configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as crs                        # coordinate systems for maps
import cartopy.feature as cfeature

plt.rcParams['figure.dpi'] = 200

BORDERS = cfeature.NaturalEarthFeature(
        scale     = '10m',
        category  = 'cultural',
        name      = 'admin_0_countries',
        edgecolor = 'gray',
        facecolor = 'none'
        )
LAND = cfeature.NaturalEarthFeature(
        'physical', 'land', '10m',
        edgecolor = 'none',
        facecolor = 'lightgrey',
        alpha     = 0.8
        )

## Plot exceedance probabilities

In [ ]:
## Data
x_train = train.values
x_vae   = x.numpy()
x_test  = test.values

lat = ds.lat
lon = ds.lon

In [ ]:
thresholds = [8,16,24]
conf = dict(
    levels = [2, 25, 50, 75, 98], 
    linestyles=['solid', 'solid', 'dashed', 'solid', 'solid'],
    colors = ["#3B6EA8", "#8FCB9B", "#000000", "#E69F00", "#F984E5"]
)

nrows, ncols = 3, len(thresholds)
fig, axs = plt.subplots(nrows=nrows, 
                        ncols=ncols,
                        sharex=True,
                        sharey=True,
                        subplot_kw={'projection': crs.PlateCarree()}, 
                        figsize=(11,8),
                       )
fig.subplots_adjust(wspace=0.1, hspace=0.1)

labels = (c for c in 'abcdefghi')
for i,threshold in enumerate(thresholds):
    p1 = 100*(x_train > threshold).mean(0)
    p2 = 100*(x_vae   > threshold).mean(0)
    p3 = 100*(x_test  > threshold).mean(0)
    
    axs[i,0].contour(lon,lat,p1,**conf)
    axs[i,1].contour(lon,lat,p2,**conf)
    axs[i,2].contour(lon,lat,p3,**conf)

    axs[i,0].set_title(f'({next(labels)}) Training dataset')
    axs[i,1].set_title(f'({next(labels)}) VAE')
    axs[i,2].set_title(f'({next(labels)}) Test dataset')

for i,threshold in enumerate(thresholds):
    label = f'Threshold: {threshold} $g/m^2$'
    for j in range(3):
        ax = axs[i,j]
        ax.text(0.05, 0.88, label, 
                fontsize=12, 
                transform=ax.transAxes)
    
for i, ax in enumerate(axs.flat):
    ax.set_extent([-22, -10, 23, 31]) # [x1,x2,y1,y2]
    ax.add_feature(LAND,zorder=0)
    ax.add_feature(BORDERS, linewidth=0.4)
    ###
    ### Enables axis labels
    ###
    row, col = np.unravel_index(i, axs.shape)
    active_labels = []
    if col == 0:
        active_labels.append('left')
    if row == nrows - 1:
        active_labels.append('bottom')
    ###
    ### Add grid lines
    ###
    gl = ax.gridlines(
        crs         = crs.PlateCarree(),
        draw_labels = active_labels,
        linewidth   = 0.5,
        color       = 'gray',
        alpha       = 0.5,
        linestyle   = '--')
    gl.xlabel_style  = {'size': 8}
    gl.ylabel_style  = {'rotation': 89, 'size': 8}